# Modelo binario — Logit condicional (elección discreta)

**La pregunta:** dado el conjunto de candidatos de un municipio, ¿cuál gana?

**El modelo correcto.** Esto es un problema de *elegir 1 entre k*, no de clasificar candidatos
sueltos. El **logit condicional** modela la probabilidad de que el candidato *i* gane como un
softmax **dentro de su contienda**:

$$P(\text{gana } i \mid \text{contienda}) = \frac{e^{\beta^\top x_i}}{\sum_{j \in \text{contienda}} e^{\beta^\top x_j}}$$

Tres propiedades que lo hacen el punto de partida correcto:
1. Respeta por construcción que gana **exactamente uno** por contienda.
2. Aprende del **contraste dentro de cada contienda** (no de probabilidades absolutas), así que
   exprime mejor las 31 contiendas.
3. Las variables **constantes dentro de la contienda** (nº de candidatos, población) **se cancelan
   solas** en la verosimilitud condicional — por eso no entran acá: no diferencian candidatos del
   mismo municipio. Tendrán su lugar en el modelo *continuo* (cuánta cuota de voto).

> **Validación:** leave-one-contest-out (dejar una contienda entera afuera). **Listón:** el baseline
> ingenuo (gana el líder en dominancia de likes) ya acierta ~71%; ese techo lo pone el dato, no el modelo.


---
## 0 — Datos y matriz de features


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from statsmodels.discrete.conditional_models import ConditionalLogit

sns.set_theme(style="whitegrid"); pd.set_option("display.max_columns", None)
%matplotlib inline
AZUL_SABANA, ROJO, GRIS = "#1A2F6F", "#FF4B4B", "#A9A9A9"


In [2]:
# ---- Carga (idéntica al maestro) ----
base_path = os.getcwd()
excel_path = os.path.join(os.path.dirname(base_path), "Colombia", "Resultados electorales.xlsx")
csv_unificado  = os.path.join(base_path, "resultados", "redes_unificadas.csv")
csv_seguidores = os.path.join(base_path, "resultados", "seguidores.csv")

df_redes      = pd.read_csv(csv_unificado, sep=";", encoding="utf-8-sig")
df_electoral  = pd.read_excel(excel_path, sheet_name="Candidatos E-26 ALC").rename(columns={"ID Candidato": "id_candidato"})
df_seguidores = pd.read_csv(csv_seguidores, sep=";", encoding="utf-8-sig").fillna(0)
for ex in ["68547", "68081"]:
    df_redes = df_redes[~df_redes["id_candidato"].str.contains(ex, na=False)]

df_completo = df_redes.merge(df_seguidores[["id_candidato","tiktok_followers","twitter_followers","facebook_followers"]], on="id_candidato", how="left")
df_completo = df_completo.merge(df_electoral[["id_candidato","Candidato","Ganador","Votos"]], on="id_candidato", how="left")
df_completo["DIVIPOLA"] = df_completo["id_candidato"].astype(str).str.split("-").str[1].str.zfill(5)
TIPOS_TODOS = ["likes","comentarios","compartidos","favoritos","fb_haha","fb_care","fb_wow","fb_sad","fb_angry"]
cols_pres = [c for c in TIPOS_TODOS if c in df_completo.columns]
df_completo["interacciones_totales"] = df_completo[cols_pres].sum(axis=1)
print(f"Candidatos: {df_completo['id_candidato'].nunique()} | Municipios: {df_completo['DIVIPOLA'].nunique()}")


Candidatos: 120 | Municipios: 31


In [3]:
# ---- prepare_lens_data + matriz de features ----
FOLLOWER_COL = {"Facebook":"facebook_followers","Twitter":"twitter_followers","TikTok":"tiktok_followers"}
def prepare_lens_data(df, metric_col, lente="dominancia", platform=None):
    data = df.copy()
    if platform is not None: data = data[data["red_social"] == platform]
    if len(data) == 0 or metric_col not in data.columns: return pd.DataFrame()
    seg = data[FOLLOWER_COL[platform]] if platform else data[list(FOLLOWER_COL.values())].sum(axis=1)
    data = data.assign(_seg=seg)
    agg = data.groupby(["DIVIPOLA","id_candidato","Candidato","Ganador"]).agg(
        metric_sum=(metric_col,"sum"), n_posts=(metric_col,"count")).reset_index()
    if lente in ("totales","dominancia"): agg["v"] = agg["metric_sum"]
    elif lente == "por_post":             agg["v"] = agg["metric_sum"]/agg["n_posts"].replace(0,np.nan)
    agg = agg.replace([np.inf,-np.inf], np.nan)
    agg["pct_metric"] = (agg["v"]/agg.groupby("DIVIPOLA")["v"].transform("sum"))*100
    return agg[agg.groupby("DIVIPOLA")["v"].transform("sum") > 0]

# Features de cuota (TODAS varían dentro de la contienda -> válidas para el logit condicional)
FEATURES = [("likes","dominancia","likes_dom"), ("comentarios","dominancia","coment_dom"),
            ("compartidos","dominancia","compart_dom"),
            ("likes","por_post","likes_pp"), ("comentarios","por_post","coment_pp"),
            ("compartidos","por_post","compart_pp")]
base = None
for tipo, lente, nombre in FEATURES:
    d = prepare_lens_data(df_completo, tipo, lente, None)[["id_candidato","DIVIPOLA","Candidato","Ganador","pct_metric"]].rename(columns={"pct_metric":nombre})
    base = d if base is None else base.merge(d[["id_candidato",nombre]], on="id_candidato", how="left")
base["y"] = (base["Ganador"] == "Sí").astype(int)
base = base.dropna(subset=[f[2] for f in FEATURES]).reset_index(drop=True)
print(f"Filas: {len(base)} | Contiendas: {base['DIVIPOLA'].nunique()} | Ganadores: {base['y'].sum()}")
base.head()


Filas: 120 | Contiendas: 31 | Ganadores: 31


,id_candidato,DIVIPOLA,Candidato,Ganador,likes_dom,coment_dom,compart_dom,likes_pp,coment_pp,compart_pp,y
0,COL-05001-001,05001,Federico Andres Gutierrez Zuluaga,Sí,61.082575,17.798919,56.042265,66.882005,17.165373,49.709993,1
1,COL-05001-004,05001,Rodolfo Andres Correa Vargas,No,0.028290,0.033868,0.407573,0.336864,0.355199,3.931537,0
2,COL-05001-010,05001,Albert Yordano Corredor Bustamante,No,24.042407,24.301482,5.241385,11.394442,10.144148,2.012321,0
3,COL-05001-011,05001,Maria Paulina Aguinaga Lezcano,No,0.666633,1.420896,0.798842,0.561978,1.055025,0.545544,0
4,COL-05001-012,05001,Juan Carlos Upegui Vanegas,No,13.605918,55.334904,36.600029,19.060318,68.276163,41.535532,0


---
## 1 — Ajuste e interpretación (sobre todo el dataset)

Arrancamos parsimoniosos: la **familia dominancia** (likes, comentarios, compartidos), que el
screening marcó como la más útil para acertar al ganador. El logit condicional **no lleva
intercepto** (la verosimilitud condicional lo elimina junto con todo lo constante por contienda).


In [4]:
FEATS = ["likes_dom", "coment_dom", "compart_dom"]   # todas varían dentro de la contienda

res = ConditionalLogit(base["y"], base[FEATS], groups=base["DIVIPOLA"]).fit(disp=0)
print(res.summary())

print("\nInterpretación (odds-ratio por +1 punto porcentual de cuota):")
for f, b in res.params.items():
    print(f"  {f:14s}: OR = {np.exp(b):.3f}  (coef {b:+.4f}, p={res.pvalues[f]:.3f})")


                  Conditional Logit Model Regression Results                  
Dep. Variable:                      y   No. Observations:                  120
Model:               ConditionalLogit   No. groups:                         31
Log-Likelihood:               -28.771   Min group size:                      2
Method:                          BFGS   Max group size:                      9
Date:                Wed, 17 Jun 2026   Mean group size:                   3.9
Time:                        23:52:49                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
likes_dom       0.0431      0.024      1.773      0.076      -0.005       0.091
coment_dom     -0.0062      0.019     -0.320      0.749      -0.044       0.032
compart_dom     0.0041      0.021      0.200      0.841      -0.036       0.044

Interpretación (odds-ratio por +1 punto porcen

Un OR > 1 significa que, dentro de la misma contienda, subir esa cuota aumenta las posibilidades
de ser el ganador. Un coeficiente no significativo (p alto) indica que, *dado* el resto, esa métrica
no aporta señal adicional — útil para podar.


---
## 2 — Validación honesta: top-1 leave-one-contest-out

Dejamos una contienda entera afuera, ajustamos con el resto, predecimos al candidato de mayor
utilidad ($\beta^\top x$) y vemos si era el ganador. Comparamos contra el baseline.


In [5]:
def top1_clogit_logo(base, feats, group_col="DIVIPOLA"):
    '''Acierto top-1 del logit condicional con leave-one-contest-out.'''
    logo = LeaveOneGroupOut()
    X = base[feats].values.astype(float); y = base["y"].values; g = base[group_col].values
    aciertos = total = 0
    for tr, te in logo.split(X, y, g):
        sc = StandardScaler().fit(X[tr])                    # escalar ayuda a converger
        Xtr, Xte = sc.transform(X[tr]), sc.transform(X[te])
        try:
            beta = np.asarray(ConditionalLogit(y[tr], Xtr, groups=g[tr]).fit(disp=0).params, dtype=float)
        except Exception:
            beta = np.zeros(Xtr.shape[1]); beta[0] = 1.0    # si no converge, degrada al baseline
        aciertos += int(y[te][np.argmax(Xte @ beta)] == 1); total += 1
    return aciertos / total

def baseline_top1(base, col, group_col="DIVIPOLA"):
    ac = tot = 0
    for g in base[group_col].unique():
        s = base[base[group_col]==g]
        ac += int(s.iloc[np.argmax(s[col].values)]["y"] == 1); tot += 1
    return ac/tot

def wilson_ci(k, n, z=1.96):
    p = k/n; d = 1+z**2/n
    c = (p+z**2/(2*n))/d; h = z*np.sqrt(p*(1-p)/n+z**2/(4*n**2))/d
    return c-h, c+h

bl   = baseline_top1(base, "likes_dom")
acc  = top1_clogit_logo(base, FEATS)
n    = base["DIVIPOLA"].nunique()
lo,hi = wilson_ci(round(acc*n), n)
print(f"BASELINE (líder likes_dom)      : {bl:.1%}")
print(f"LOGIT CONDICIONAL (LOGO)        : {acc:.1%}  (IC 95%: {lo:.0%}–{hi:.0%})")
print(f"Diferencia vs baseline          : {acc-bl:+.1%}")
print(f"\nNota: con n={n} contiendas el IC es ancho; diferencias de 1-2 contiendas son ruido.")


BASELINE (líder likes_dom)      : 71.0%
LOGIT CONDICIONAL (LOGO)        : 64.5%  (IC 95%: 47%–79%)
Diferencia vs baseline          : -6.5%

Nota: con n=31 contiendas el IC es ancho; diferencias de 1-2 contiendas son ruido.


In [ ]:
# --- Comparar distintos conjuntos de features (cuál set predice mejor) ---
sets = {"Solo likes_dom (≈baseline)": ["likes_dom"],
        "Familia dominancia": ["likes_dom","coment_dom","compart_dom"],
        "Familia por_post":   ["likes_pp","coment_pp","compart_pp"],
        "Dominancia + por_post": [f[2] for f in FEATURES]}
filas = [{"features": "BASELINE (argmax likes_dom)", "top1_%": round(bl*100,1)}]
for nombre, fs in sets.items():
    filas.append({"features": nombre, "top1_%": round(top1_clogit_logo(base, fs)*100, 1)})
tabla = pd.DataFrame(filas).sort_values("top1_%", ascending=False).reset_index(drop=True)
tabla


---
## 3 — Diagnóstico: ¿dónde falla?

Las contiendas donde el líder digital no fue el ganador son las que ningún modelo basado solo en
interacciones puede arreglar (Bogotá, donde Bolívar dominó las redes pero no ganó, es el caso clásico).


In [6]:
nombres = df_completo[["DIVIPOLA"]].drop_duplicates()
nombres = nombres.merge(df_completo[["DIVIPOLA","Municipio"]].drop_duplicates(), on="DIVIPOLA", how="left") \
    if "Municipio" in df_completo.columns else nombres.assign(Municipio=nombres["DIVIPOLA"])

fallos = []
for g in base["DIVIPOLA"].unique():
    s = base[base["DIVIPOLA"]==g]
    lider = s.iloc[np.argmax(s["likes_dom"].values)]
    if lider["y"] != 1:
        gana = s[s["y"]==1]["Candidato"].values
        muni = nombres[nombres["DIVIPOLA"]==g]["Municipio"].values
        fallos.append({"municipio": muni[0] if len(muni) else g,
                       "lider_digital": lider["Candidato"], "lider_likes_%": round(lider["likes_dom"],1),
                       "ganador_real": gana[0] if len(gana) else "?", "n_cand": len(s)})
print(f"Contiendas donde el líder digital NO ganó: {len(fallos)} de {base['DIVIPOLA'].nunique()}")
pd.DataFrame(fallos)


Contiendas donde el líder digital NO ganó: 9 de 31


,municipio,lider_digital,lider_likes_%,ganador_real,n_cand
0,05151,Rosa Maria Acevedo Jaramillo,77.9,Diego Leon Torres Sanchez,3
1,08433,Jose Joao Herrera Iranzo,55.7,Alcira Paola Sandoval Ibañez,3
2,11001,Gustavo Bolivar Moreno,50.1,Carlos Fernando Galan Pachon,9
3,19001,Diana Nelly Fuentes Meneses,44.5,Juan Carlos Muñoz Bravo,3
4,23001,Natalia Eugenia Lopez Fuentes,69.5,Hugo Fernando Kerguelen Garcia,2
5,41001,Jorge Andres Gechem Artunduaga,35.1,German Casagua Bonilla,6
6,63001,Stefany Gomez Murillo,79.0,James Padilla Garcia,3
7,66170,Tatiana Lopez Saldarriaga,56.3,Roberto Jimenez Naranjo,3
8,70001,Diego Mercado Sanabria,36.6,Yahir Fernando Acuña Cardales,3


---
## Notas y siguiente paso

- **El techo es el dato, no el modelo.** Si el logit condicional ronda el baseline (~71%), es
  porque las ~9 excepciones son casos donde las interacciones simplemente no rastrean el voto.
- **Por qué este modelo sí es defendible:** es el modelo estadístico estándar para "elegir 1 entre
  k", respeta la estructura, da coeficientes interpretables y es el hermano discreto del modelo de
  cuota de voto. Ante el jurado, "ajusté un logit condicional" es sólido; "le metí una red neuronal
  a 120 filas" no.
- **Reporta siempre el acierto con su IC** y al lado del baseline.
- **Siguiente:** el modelo **continuo** (¿qué cuota de voto?), donde la población, la abstención y
  la categoría DNP sí entran como predictores directos, y el logit fraccional / beta es el natural.


# **el nowcaster ingenuo acierta 71% contra un azar de 30%**